# Creation of a database for index and ETF comparision

### This sheet's role would be to clearly identify which CIFSC categories are associated with which ETF/Index we can get from YFinance. Once we have downloaded such data, we would be able to compare their monthly returns with the one in the CIFSC data sheet

In [10]:
## Import necessary packages
import yfinance as yf
import pandas as pd

## This is the grid with the CIFSC categories associated with their respective index / etf

 Consolidated Category       | ETF/Index Ticker | Description                                                                 |
|----------------------------|------------------|-----------------------------------------------------------------------------|
| **Sector Equity**          | VFH              | **Vanguard Financials ETF** - Tracks the MSCI U.S. Investable Market Financials Index, providing balanced exposure to financial sector equities across North America, covering diverse industries within financials. |
| **Global Equity**          | VXUS             | **Vanguard Total International Stock ETF** - Provides broad exposure to non-U.S. developed and emerging markets equities. |
| **Miscellaneous**          | AOM              | **iShares Core Moderate Allocation ETF** - A balanced fund with approximately 40% global equities and 60% global bonds, offering a mix of public equity and public debt worldwide. |
| **Balanced - Tactical**    | SWAN             | **Amplify BlackSwan Growth & Treasury Core ETF** - A tactical bond fund combining long U.S. Treasuries with S&P 500 call options for risk-managed, tactical exposure. |
| **U.S./North American Equity** | SPY           | **SPDR S&P 500 ETF Trust** - Tracks the S&P 500, representing large-cap U.S. equities. Highly liquid and a proxy for North American equity. |
| **Target Date**            | VTHRX            | **Vanguard Target Retirement 2030 Fund** - A mutual fund tracking a balanced index, suitable for target-date strategies. Available via **yfinance**. |
| **Canadian Fixed Income**  | XBB.TO           | **iShares Core Canadian Universe Bond Index ETF** - Tracks Canadian investment-grade bonds, a close match for Canadian fixed income. Listed on TSX. |
| **Alternative**            | PFF              | **iShares Preferred and Income Securities ETF** - Tracks preferred stocks and hybrid securities, offering high exposure to alternative investments resembling private equity and debt characteristics. |
| **Emerging/Asia Equity**   | EEM              | **iShares MSCI Emerging Markets ETF** - Tracks emerging markets, with significant Asia exposure, suitable for emerging/Asia equity. |
| **Balanced - Global**      | AOA              | **iShares Core Aggressive Allocation ETF** - A global balanced ETF with 80% stocks (domestic and international) and 20% bonds. |
| **Canadian Equity**        | XIU.TO           | **iShares S&P/TSX 60 Index ETF** - Tracks the top 60 Canadian companies by market cap, a standard for Canadian equity exposure. Listed on TSX. |
| **Money Market**           | BIL              | **SPDR Bloomberg 1-3 Month T-Bill ETF** - Tracks short-term U.S. Treasury bills, a close proxy for money market funds with high liquidity. |
| **Europe Equity**          | VGK              | **Vanguard FTSE Europe ETF** - Tracks European developed market equities, a direct match for Europe-focused equity funds. |
| **Global Fixed Income**    | BNDW             | **Vanguard Total World Bond ETF** - Provides exposure to global investment-grade bonds, suitable for global fixed income. |
| **Commodity**              | DBC              | **Invesco DB Commodity Index Tracking Fund** - Tracks a diversified basket of commodities (e.g., crude oil, metals, agriculture), providing broad commodity exposure beyond gold. |
| **Balanced - Canadian**    | XBAL.TO          | **iShares Core Balanced ETF Portfolio** - A Canadian-listed ETF with a mix of Canadian equities and bonds, aligning with balanced Canadian funds. |

## Get ticker monthly data and sort it by ticker and date
## Delete price open, high low and volume columns



In [11]:
# Define the tickers
tickers = ['VFH', 'VXUS', 'AOM', 'SWAN', 'SPY', 'VTHRX', 'XBB.TO', 'PFF', 'EEM', 'AOA', 'XIU.TO', 'BIL', 'VGK', 'BNDW', 'DBC', 'XBAL.TO']

# Download the data
data2 = yf.download(tickers, start='2016-01-01', end='2025-02-01', interval='1mo', group_by='ticker')

# Reshape the data so each ticker has its own column
data2 = data2.stack(level=0).reset_index()
data2 = data2.drop(columns=['Open', 'High', 'Low', 'Volume'])
# Save the data to a CSV file
data2.to_csv('data2.csv', index=False)

data3 = data2.sort_values(by=['Ticker', 'Date']).reset_index(drop=True)	
# Save the reshaped data to a CSV file  
data3.to_csv('data3.csv', index=False)

print(data3.head())

YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  16 of 16 completed

Price       Date Ticker      Close
0     2016-01-01    AOA  34.324764
1     2016-02-01    AOA  34.085175
2     2016-03-01    AOA  36.185551
3     2016-04-01    AOA  36.409172
4     2016-05-01    AOA  36.648750



C:\Users\AzureVirtualDesktopU\AppData\Local\Temp\ipykernel_3420\1690844568.py:8: FutureWarning: The previous implementation of stack is deprecated and will be removed in a future version of pandas. See the What's New notes for pandas 2.1.0 for details. Specify future_stack=True to adopt the new implementation and silence this warning.
  data2 = data2.stack(level=0).reset_index()


In [12]:
print(data3.columns)

Index(['Date', 'Ticker', 'Close'], dtype='object', name='Price')


## get monthly returns per ticker

In [13]:
import duckdb

duckdb.register("data3", data3)
# Register your Pandas DataFrame as a DuckDB table
duckdb.register("data4", data3)

query = """
SELECT 
  *,
  Close / FIRST_VALUE(Close) OVER (PARTITION BY Ticker ORDER BY Date) AS CloseNormalized,
  (Close - lag(Close) OVER (PARTITION BY Ticker ORDER BY Date)) / lag(Close) OVER (PARTITION BY Ticker ORDER BY Date) AS PercentMonthlyChange
FROM data3;
"""

# Execute the query and convert the result to a Pandas DataFrame.
data4 = duckdb.query(query).to_df()
data4.head()


,Date,Ticker,Close,CloseNormalized,PercentMonthlyChange
0,2016-01-01,EEM,25.079803,1.000000,NaN
1,2016-02-01,EEM,24.874699,0.991822,-0.008178
2,2016-03-01,EEM,28.098896,1.120379,0.129618
3,2016-04-01,EEM,28.213749,1.124959,0.004087
4,2016-05-01,EEM,27.171839,1.083415,-0.036929


### Upload the consolidated data by CIFSC categories 

In [14]:
ConsolidatedMutualFundByGroup = pd.read_csv('ConsolidatedMutualFund.csv')
ConsolidatedMutualFundByGroup.head()
ConsolidatedMutualFundByGroup["Month Number"].values

array([  1,   2,   3, ..., 103, 104, 105], shape=(1780,))

### Add the month number that correspond to the right number to data4 = ETF database

In [15]:


# Assuming data4 is your DataFrame and "Date" is in datetime format
data4['Date'] = pd.to_datetime(data4['Date'])  # Ensure "Date" is in datetime format

# Define the reference date
reference_date = pd.Timestamp('2016-01-01')

# Calculate the "Month Number"
data4['Month Number'] = ((data4['Date'].dt.year - reference_date.year) * 12 +
                         (data4['Date'].dt.month - reference_date.month) + 1)

# Display the updated DataFrame
print(data4)

           Date  Ticker      Close  CloseNormalized  PercentMonthlyChange  \
0    2016-01-01     EEM  25.079803         1.000000                   NaN   
1    2016-02-01     EEM  24.874699         0.991822             -0.008178   
2    2016-03-01     EEM  28.098896         1.120379              0.129618   
3    2016-04-01     EEM  28.213749         1.124959              0.004087   
4    2016-05-01     EEM  27.171839         1.083415             -0.036929   
...         ...     ...        ...              ...                   ...   
1673 2024-09-01  XBB.TO  27.921244         1.158850              0.018038   
1674 2024-10-01  XBB.TO  27.645290         1.147397             -0.009883   
1675 2024-11-01  XBB.TO  28.124992         1.167306              0.017352   
1676 2024-12-01  XBB.TO  27.919207         1.158765             -0.007317   
1677 2025-01-01  XBB.TO  28.243877         1.172240              0.011629   

      Month Number  
0                1  
1                2  
2           